In [ ]:
# Detect if running inside Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os
    if not os.path.exists("dye-identification-spectral-unmixing"):
        !git clone https://github.com/neilhkim/dye-identification-spectral-unmixing.git
    %cd dye-identification-spectral-unmixing# Clone repository if running in Colab
import os

if not os.path.exists("data"):
    !git clone https://github.com/neilhkim/dye-identification-spectral-unmixing.git
    %cd dye-identification-spectral-unmixing

# Countable Labs - Assessment for Sr/Staff Engineer, Computational Biology & Imaging

*Prompt 2 — Dye Identification from Spectral Intensity Data*

- Author: Hyuneil (Neil) Kim

# Problem Setup

Each PCR compartment produces fluorescence measurements across multiple detection channels.

A compartment may contain:

- no dye
- one dye
- two dyes

Because dye spectra overlap, each channel receives a mixture of dye signals.

---

## Provided Data

- Observations

$$
Y \in \mathbb{R}^{N \times C}
$$

N compartments × C channels (here C (num of channels) = 8)

- Reference spectra

$$
R \in \mathbb{R}^{C \times D}
$$

Each column represents the spectral signature of one dye measured from controls.

(here D (num of dyes) = 8)

---

## Goal

Assign the dye(s) present in each compartment using the multi-channel measurements.

As such, below is how I am tackle the problem:

## Table of Contents

1. [Signal Model](#1-signal-model)
2. [Key Challenges](#2-key-challenges)
3. [Algorithm Overview](#3-algorithm-overview)
4. [Loading Data and Visualizing Linear Superposition](#4-loading-data-and-visualizing-the-linear-superposition)
5. [Characterizing the Actual Noise Profile](#5-characterizing-the-actual-noise-profile-epsilon)
   - [5.1 Analysis of Empty Compartments](#51-analysis-of-empty-compartments)
   - [5.2 Sensor Saturation Check](#52-sensor-saturation-check)
6. [Diagnosing the Inverse Problem](#6-diagnosing-the-inverse-problem-x--r-1y---epsilon)
   - [6.1 Impact of Increasing Noise on Naive Inversion](#61-the-impact-of-increasing-noise-on-naive-inversion-simulation-approach)
   - [6.2 Addressing the Failure Modes](#62-addressing-the-failure-modes)
7. [Exhaustive Model Selection with BIC](#7-exhaustive-model-selection-with-bic)
8. [Goodness-of-Fit Safeguard](#8-goodness-of-fit-r2-safeguard)
9. [Full Dataset Processing](#9-the-complete-algorithm--execution)
10. [Final Report](#10-final-report)
11. [Diagnostic Analysis: Empty vs Signal](#11-diagnostic-analysis-empty-vs-signal)
12. [Interpretation: Why “Empty” Really Is Empty](#12-interpretation-why-empty-really-is-empty)
13. [Summary](#13-summary)

----

## 1. Signal Model

We assume the measured spectrum is a linear combination of dye spectra.

$$
y = R x + \epsilon
$$

Where

- $y$ : observed 8-channel signal
- $R$ : reference dye matrix
- $x$ : dye intensities
- $\epsilon$ : noise

Physical constraints of the assay:

- $x \ge 0$ (fluorescence cannot be negative)
- each compartment contains **at most two dyes**

This produces a constrained spectral unmixing problem.

Fundamentally, the task is to solve the inverse problem: compute the true physical state $x$ given an observation $y$ and the reference basis $R$.

## 2. Key Challenges

### Spectral overlap

Dye spectra overlap across channels, so channel intensities cannot be interpreted independently.

We therefore fit the **entire spectral vector simultaneously**.

### Weak signal vs background

Empty compartments still produce detector noise.

We detect background using the signal magnitude

$$
\|y\|
$$

Very weak signals are classified as background before attempting dye identification.

### Confidence in assignments

We evaluate model quality using

- **Bayesian Information Criterion (BIC)** — penalizes unnecessary dyes
- **$R^2$ goodness of fit**

Low-fit compartments are flagged as anomalies.


## 3. Algorithm Overview

For each compartment:

1. **Background detection**

   If signal magnitude is below a threshold → classify as background.

2. **Enumerate possible dye models**

   - 8 single-dye models
   - 28 two-dye models

3. **Fit each model**

   Solve

$$
\min_{x \ge 0} \|y - R x\|^2
$$

   using non-negative least squares.

4. **Model selection**

   Choose the model with lowest

$$
BIC = n \ln(RSS/n) + k \ln(n)
$$, 

where $n$ is the number of channels, $RSS$ is the residual sum of squares, and $k$ is the assumed number of dyes.

5. **Confidence check**

   Reject models with low

$$
R^2
$$

This produces the final dye assignment.

## 4. Loading Data and Visualizing the Linear Superposition
To make this mathematical framework concrete, I will load the provided datasets and simulate a 2-dye mixture. Suppose a compartment contains Dye 1 and Dye 4 with an intensity layout of $x_1 = 0.8$ and $x_4 = 0.2$. All other elements in $x$ are zero.Below is a small visualization of the linear combination

In [ ]:
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import nnls
from scipy.ndimage import gaussian_filter1d

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (10, 5), 'figure.dpi': 100, 'axes.titlesize': 14})

# ---------------------------------------------------------
# 1. Load Reference Spectra from CSV
# ---------------------------------------------------------
ref_path = "./data/reference_spectra.csv"

try:
    ref_df = pd.read_csv(ref_path, sep=None, engine="python")
    if not str(ref_df.columns[0]).strip().startswith("Dye_"):
        ref_df = pd.read_csv(ref_path, skiprows=1, sep=None, engine="python")
except Exception:
    ref_df = pd.read_csv(ref_path, skiprows=1)

ref_df = ref_df.iloc[:, :8].apply(pd.to_numeric, errors="coerce").dropna(how="all")
R_raw = ref_df.to_numpy(dtype=float)

if R_raw.shape != (8, 8):
    raise ValueError(f"Expected 8x8 reference spectra, got {R_raw.shape}")

R_norm = R_raw / R_raw.max(axis=0)  
print(f"Reference Matrix R loaded from CSV (normalized):\n{R_norm.round(2)}\n")

print()


# ---------------------------------------------------------
# 3. Simulate and Visualize the Physical Superposition
# ---------------------------------------------------------
x_vis = np.array([0.8, 0.0, 0.0, 0.2, 0.0, 0.0, 0.0, 0.0])
np.random.seed(42)
y_vis = np.clip(R_norm @ x_vis + np.random.normal(0, 0.03, 8), 0, None)

print(f"Input Intensity Layout (x):\n{x_vis}")

plt.figure(figsize=(8, 4))
plt.plot(range(1, 9), R_norm[:,0], '--o', alpha=0.6, label='Dye 1 readout (normalized); Column 1 of R')
plt.plot(range(1, 9), R_norm[:,3], '--s', alpha=0.6, label='Dye 4 readout (normalized); Column 4 of R')
plt.plot(range(1, 9), y_vis, '-D', lw=2.5, color='crimson', label='Observed Signal (Dye 1: 0.8, Dye 4: 0.2, with noise)')
plt.xlabel("Detection Channel")
plt.ylabel("Readout")
plt.legend(loc="right")
plt.title(r"Example Physical Superposition: $y = Rx + \epsilon$")
plt.tight_layout()
plt.show()

The spectral overlap from the reference spectra matrix is obviously present.
An important focus of this project is to call dyes correctly despite the spectral overlap.

---

## 5. Characterizing the Actual Noise Profile ($\epsilon$)

To accurately solve the inverse problem ($x \approx R^{-1}(y - \epsilon)$), I must understand the true nature of $\epsilon$ in this specific optical instrument. 
- Is it static Gaussian noise? 
- Is it photon shot noise? 
- Is there another noise source to be identified?

I will interrogate the raw `intensity_data.csv` to answer these questions:

1.  **Analysis of empty compartments:** I will calculate the $L_2$-norm ($||y||_2$) of all compartments. By plotting the histogram, I can identify the "empty" compartments (where $y \approx \epsilon$) and examine the noise characterstics.
2.  **Sensor Saturation:** I will check if the hardware has a hard clipping limit.

---

### 5.1 Analysis of empty compartments

In [ ]:
# ====================================================================
# Characterizing the actual noise in intensity_data.csv
# ====================================================================
from scipy.signal import find_peaks


# ---------------------------------------------------------
# Load Observational Intensity Data from CSV
# ---------------------------------------------------------
data_path = "./data/intensity_data.csv"
try:
    Y_real = pd.read_csv(data_path).to_numpy(dtype=float)
except Exception:
    # Safe fallback if file is missing locally
    print("WARNING: intensity_data.csv not found.")
    Y_real = np.zeros((10, 8)) 

print(f"Observed Data Y loaded. Shape: {Y_real.shape}\n")


# 1. Identify Empty Compartments to estimate static noise
magnitudes = np.linalg.norm(Y_real, axis=1)

# Generate a smoothed histogram to naturally find the separation valley
counts, bin_edges = np.histogram(magnitudes, bins=100)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
# Use a small Gaussian smoothing (sigma=2) to suppress bin-to-bin sampling noise
# while preserving the two main modes (empty vs signal) and their separating valley.
smoothed_counts = gaussian_filter1d(counts, sigma=2)

# Algorithmically locate the empty peak, the signal peak, and the valley between them
# We use topological peak finding instead of an arbitrary fraction of the array.
# 'prominence' filters out tiny ripples and only finds structurally significant peaks.
# Pad to allow finding peaks at the boundaries (e.g., bin 0)
padded_counts = np.pad(smoothed_counts, (1, 1), mode='constant', constant_values=0)
peaks, _ = find_peaks(padded_counts, prominence=np.max(smoothed_counts) * 0.02)
peaks = peaks - 1 # Adjust indices back to original array

if len(peaks) >= 2:
    # The noise mode occurs at the lowest magnitude, so it physically must be the first peak.
    noise_peak_idx = peaks[0] 
    
    # The actual signal might have multiple peaks (if certain dye combos are very common),
    # so we find the global maximum of all the remaining signal peaks.
    signal_peaks = peaks[1:]
    signal_peak_idx = signal_peaks[np.argmax(smoothed_counts[signal_peaks])]
    
    # The fundamental division is the stable minimum count between these two populations
    valley_slice = smoothed_counts[noise_peak_idx:signal_peak_idx]
    valley_idx = int(np.argmin(valley_slice)) + noise_peak_idx
else:
    # Safe fallback if peak finding fails (e.g., if the histogram is too noisy or unimodal)
    # The given data shows clear bimodality, so this is unlikely, but it is written to avoid crashes.
    noise_peak_idx = int(np.argmax(smoothed_counts))
    valley_idx = np.argmin(smoothed_counts)
    signal_peak_idx = len(smoothed_counts) - 1

noise_threshold = bin_centers[valley_idx]

# --- Visualization: Peaks and Valleys ---
plt.figure(figsize=(8, 4))
plt.hist(magnitudes, bins=100, alpha=0.5, color='gray', label='Magnitude Histogram')
plt.plot(bin_centers, smoothed_counts, color='blue', lw=2, label='Smoothed Histogram')
plt.axvline(bin_centers[noise_peak_idx], color='red', linestyle='--', label='Noise Peak')
plt.axvline(bin_centers[signal_peak_idx], color='green', linestyle='--', label='Signal Peak')
plt.axvline(noise_threshold, color='black', linestyle=':', lw=2, label=f'Valley Threshold ({noise_threshold:.0f})')
plt.title("Magnitude Distribution: Signal vs Empty Compartments")
plt.xlabel("L2 Norm Magnitude")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()



It looks reasonable to consider all compartments L2 Norm Magnitude below the threshold to be empty. I will now show the distribution of each channel readout from those empty compartments.

In [ ]:

# Identify Empty Compartments
empty_mask = magnitudes < noise_threshold
Y_empty = Y_real[empty_mask]

# Extract empirical Static Gaussian parameters from the identified empty compartments
static_mean = np.mean(Y_empty, axis=0) # mean intensity for each channel where compartment is determined to be empty
static_std = np.std(Y_empty, axis=0) # standard deviation for each channel in empty compartments
static_var = np.var(Y_empty, axis=0) # variance for each channel

# --- Visualization: Static Background Noise ---
fig, axes = plt.subplots(2, 4, figsize=(7, 4), sharex=True, sharey=True)
for i, ax in enumerate(axes.flatten()):
    ax.hist(Y_empty[:, i], bins=30, color='gray', alpha=0.7)
    ax.set_title(f"Channel {i+1}")
    if i >= 4:
        ax.set_xlabel("Intensity")
    if i % 4 == 0:
        ax.set_ylabel("Frequency")

n_empty = Y_empty.shape[0]
plt.suptitle(f"Static Background Noise Distribution per Channel (Empty Compartments, n={n_empty})")
plt.tight_layout()
plt.show()

# --- Visualization: Variance vs Mean ---
plt.figure(figsize=(6, 4))
plt.scatter(static_mean, static_var, color='black', s=50, zorder=5)

for i in range(8):
    plt.text(static_mean[i] + max(static_mean) * 0.01, static_var[i], f'Ch {i+1}', va='center')

# Linear fit: variance = m * mean + b
m, b = np.polyfit(static_mean, static_var, 1)
x_fit = np.linspace(static_mean.min(), static_mean.max(), 100)
y_fit = m * x_fit  + b
plt.plot(x_fit, y_fit, color='crimson', lw=2, label=f'Linear fit: y={m:.2f}x+{b:.2f}')

plt.title("Variance vs Mean per Channel (Empty Compartments)")
plt.xlabel("Mean Intensity")
plt.ylabel("Variance")
plt.legend()
plt.tight_layout()
plt.show()

This is quite interesting.
I expected to find read (e.g. electronic noise floor), which would not show Poisson-noise characterstics.
However, the relationship between the variance and the mean readout in each channel is disimilar to read noise or electric noise floor. Rather, because the variance and the mean has a strong linear relationship, it strongly signals for Poisson (shot) noise, perhaps stray light being the origin.

If the channel readouts were photon counts, the slope would have been 1.0. However, since the camera has gain, the slope likely represents the gain. 

However, the negative y-intercept is difficult to interpret. If it were positive, it might have indicated a different source of noise on top of the shot noise. But with a negative intercept, is there a static gaussian noise or not? It's hard to tell from the negative intercept.

I will *stop* analyzing the noise characteristics of empty compartments and move on because I cannot conclude anything much from it. Next, I will examine if saturation is occuring in the readouts.

---

### 5.2 Sensor saturation check

We can now check for signal saturation, specifically examining the readouts hitting the 16-bit sensor limit of 65535. I see a lot of readout values exactly on 65535 and nothing above it as below: 

In [ ]:
# Load necessary libraries and data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Load the intensity data
intensity_data = pd.read_csv('data/intensity_data.csv')

# Display basic info about the data
print("Data shape:", intensity_data.shape)
print("\nData info:")
print(intensity_data.describe())

print("\nUnique values around 65535:")
# Check for values at or near 65535
max_values = []
for column in intensity_data.columns:
    max_val = intensity_data[column].max()
    max_values.append(max_val)
    count_65535 = (intensity_data[column] == 65535).sum()
    count_above_65535 = (intensity_data[column] > 65535).sum()
    print(f"{column}: Max={max_val}, Count of 65535={count_65535}, Count above 65535={count_above_65535}")

print(f"\nMaximum value across all channels: {max(max_values)}")
print(f"Number of channels with max value 65535: {sum([val == 65535 for val in max_values])}")

In [ ]:
# Create comprehensive histograms for all channels
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
fig.suptitle('Histogram of Values for Each Channel\nShowing Distribution and 65535 Saturation', fontsize=16)

# Flatten axes for easier iteration
axes_flat = axes.flatten()

for i, column in enumerate(intensity_data.columns):
    ax = axes_flat[i]
    
    # Get data for this channel
    data = intensity_data[column]
    
    # Create histogram
    n, bins, patches = ax.hist(data, bins=100, alpha=0.7, color='skyblue', edgecolor='black')
    
    # Highlight bins containing 65535
    for j, (left, right) in enumerate(zip(bins[:-1], bins[1:])):
        if left <= 65535 <= right:
            patches[j].set_facecolor('red')
            patches[j].set_alpha(0.8)
    
    # Add statistics
    max_val = data.max()
    count_65535 = (data == 65535).sum()
    count_above_65535 = (data > 65535).sum()
    total_count = len(data)
    
    # Set title and labels
    ax.set_title(f'{column}\nMax: {max_val} | 65535 count: {count_65535} ({count_65535/total_count*100:.1f}%)\nAbove 65535: {count_above_65535}')
    ax.set_xlabel('Intensity Value')
    ax.set_ylabel('Frequency')
    
    # Add vertical line at 65535
    ax.axvline(x=65535, color='red', linestyle='--', linewidth=2, alpha=0.8, label='65535 threshold')
    
    # Set x-axis to show full range
    ax.set_xlim(0, max(70000, max_val + 1000))
    
    # Add grid
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

Thus, it is highly likely there's saturation at 65536.

Now, I'll quantify how many compartments experienced the "saturation".

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data from your results
counts_all = [9358, 228, 374, 40]
counts_ne = [8311, 228, 374, 40]
n_non_empty = sum(counts_ne)

# Labels updated per your request (3+ changed to 3)
labels = ['0 saturated', '1 channel', '2 channels', '3 channels']

# Higher contrast, professional palette
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 10))

def draw_clean_pie(ax, data, title):
    total = sum(data)
    wedges, _ = ax.pie(
        data,
        startangle=90,
        counterclock=False,
        colors=colors,
        wedgeprops={'edgecolor': 'w', 'linewidth': 1.5}
    )

    # Significant padding (70) keeps the title safely above any label boxes
    ax.set_title(title, pad=70, fontsize=15, weight='bold')

    # Box and line styling
    bbox_props = dict(boxstyle="round,pad=0.3", fc="white", ec="white", lw=0.7, alpha=0.9)
    kw = dict(arrowprops=dict(arrowstyle="-", color="gray", lw=1.2),
              bbox=bbox_props, zorder=5, va="center")

    # Trackers for staggered vertical positioning
    y_occupied_left = []
    y_occupied_right = []

    for i, p in enumerate(wedges):
        if data[i] == 0:
            continue
            
        # Get the center point of the wedge
        ang = (p.theta2 - p.theta1) / 2. + p.theta1
        y = np.sin(np.deg2rad(ang))
        x = np.cos(np.deg2rad(ang))

        x_sign = 1 if x >= 0 else -1
        ha = "left" if x_sign == 1 else "right"
        
        # Position labels slightly outside the pie
        y_pos = 1.15 * y
        x_pos = 1.4 * x_sign
        
        # Hard cap: prevent labels from floating into the title space
        if y_pos > 1.05:
            y_pos = 1.05

        # Collision avoidance: shift labels if they overlap on the same side
        buffer = 0.25
        occupied_list = y_occupied_right if x_sign == 1 else y_occupied_left
        
        attempts = 0
        while any(abs(y_pos - occupied) < buffer for occupied in occupied_list) and attempts < 10:
            # Shift away from center
            y_pos += buffer if y_pos >= 0 else -buffer
            attempts += 1
        occupied_list.append(y_pos)

        # Connection line geometry
        connectionstyle = f"angle,angleA=0,angleB={ang}"
        kw["arrowprops"].update({"connectionstyle": connectionstyle})

        pct = data[i] / total * 100
        text = f"{labels[i]}\n{pct:.1f}% (n={data[i]:,})"

        ax.annotate(text, xy=(x, y), xytext=(x_pos, y_pos),
                    horizontalalignment=ha, size=9, **kw)

# Render both plots
draw_clean_pie(ax1, counts_all, 'Saturation per Compartment (All 10,000)')
draw_clean_pie(ax2, counts_ne, f'Saturation per Compartment (Non-empty Only, n={n_non_empty:,})')

plt.tight_layout()
plt.subplots_adjust(hspace=0.5) # Increased vertical spacing between subplots
plt.show()

So there are some saturations resulting in clipping. It happens for about 6.4% of all compartments, and about 7.2% of non-empty compartments.

Having looked at the noise characteristics of empty compartments and error coming from saturations, we will move on to the next section.

---

## 6. Diagnosing the Inverse Problem: $x = R^{-1}(y - \epsilon)$

With a better understanding of the noise characteristics, I will evaluate mathematical approaches to compute $x$. 

Firstly, I will check whether $R$ is invertible by evaluating its rank, determinant, and most importantly, its **Condition Number**. A condition number below 100 means the matrix is well-conditioned and won't severely amplify measurement noise during inversion.

In [ ]:
# 1. Check Rank
rank = np.linalg.matrix_rank(R_norm)
is_full_rank = rank == R_norm.shape[0]

# 2. Check Determinant
det = np.linalg.det(R_norm)

# 3. Check Condition Number (Ideally < 100 for stable inversion)
cond_num = np.linalg.cond(R_norm)

print(f"Matrix Rank:      {rank} (Full Rank: {is_full_rank})")
print(f"Determinant:      {det:.2e}")
print(f"Condition Number: {cond_num:.2f}")

if is_full_rank and cond_num < 100:
    print("\nResult: R_norm is invertible and numerically stable for inversion.")
else:
    print("\nResult: R_norm is singular or ill-conditioned.")

A condition number of ~67.5 is good. It guarantees that, in the absolute worst-case scenario, 1% measurement noise in $y$ will amplify to at most a 67.5% error in the solution $x$, and in most cases it will be a lot less.

<div style="padding: 14px 16px; border-left: 8px solid #da18d3; border-radius: 8px; font-size: 1.15em; line-height: 1.4;">
    <strong style="font-size: 1.25em;">Important:</strong><br/>
    <strong>Even though there is spectral overlap between dyes, the strong numerical stability of inverting <em>R</em> means dye contributions can be reliably recovered from the 8‑channel readout (within noise limits). This also means that truly empty compartments remain distinguishable from any real dye signal: we can know empty is empty and not signal.</strong>
</div>


---


### 6.1 The Impact of Increasing Noise Naive Inversion (Simulation approach)
To test if standard matrix inversion ($\hat{x} = R^{-1}y$) holds up in practice, I will simulate a 1-dye compartment and progressively increase the complexity of the noise using the exact parameters I just extracted from the dataset:

1. **Static Gaussian Noise** (Baseline + Read Noise) 
2. **Clipping** (Where large reads get cut off at 65535).

---

#### 1. The Effect of Static Gaussian Noise

In [ ]:
R_inv = np.linalg.inv(R_norm)

base_intensity = 25000.0

# Helper to simulate 8 cases and get 8x8 inferred matrix
def simulate_8_cases(noise_fn, intensity=base_intensity):

    # Columns represent the 8 true cases (Dye 1 to 8)
    X_true = np.eye(8) * intensity

    Y_clean = R_norm @ X_true
    Y_noisy = noise_fn(Y_clean)

    X_hat = R_inv @ Y_noisy

    return X_true, X_hat


np.random.seed(42)

# ==========================================================
# Static Noise Case (Single Scenario)
# ==========================================================

mean = 800
sd = 500

def static_noise(Y):
    return np.clip(Y + np.random.normal(mean, sd, Y.shape), 0, 65535)

_, X_hat = simulate_8_cases(static_noise)

fig, ax = plt.subplots(figsize=(6, 4))
fig.suptitle("Naive Inference with Static Noise (μ=500, σ=200)", fontsize=14)

# X_hat has inferred dyes on rows, true dyes on columns
im = ax.imshow(X_hat, cmap='RdBu_r', vmin=-35000, vmax=35000)

ax.set_xlabel("True Dye (Ground Truth)")
ax.set_ylabel("Inferred Dye Component")

ax.set_xticks(range(8))
ax.set_yticks(range(8))
ax.set_xticklabels(range(1, 9))
ax.set_yticklabels(range(1, 9))

ax.set_xticks(np.arange(-0.5, 8, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 8, 1), minor=True)

ax.grid(which="minor", color="w", linestyle='-', linewidth=1)
ax.grid(which="major", visible=False)
ax.tick_params(which="minor", bottom=False, left=False)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Inferred Intensity ($\\hat{x}$)")

plt.tight_layout()
plt.show()

The static noise harms the dye-calling somewhat. 
Note that the negative intensitiy predictions, which are physically not possible.

---

Next, we examine the saturation (clipping effect)

In [ ]:


# ====================================================================
# 2. Saturation (Clipping) Effect for each of 8 dye cases
# ====================================================================
def clipping_noise(Y):
    # Hardware sensor hard clipping at max 16-bit unsigned int
    return np.clip(Y, 0, 65535)

high_intensity = 80000.0  # Deliberately causes clipping (100k > 65535)
X_true_clip, X_hat_clip = simulate_8_cases(clipping_noise, intensity=high_intensity)

fig, ax = plt.subplots(figsize=(6, 4))
fig.suptitle(f"Naive Inference under Saturation\n(Hard clip at 65535, True Intensity={high_intensity:.0f})", fontsize=14)

im = ax.imshow(X_hat_clip, cmap='RdBu_r', vmin=-110000, vmax=110000)
ax.set_xlabel("True Dye (Ground Truth)")
ax.set_ylabel("Inferred Dye Component")

ax.set_xticks(range(8))
ax.set_yticks(range(8))
ax.set_xticklabels(range(1, 9))
ax.set_yticklabels(range(1, 9))
ax.set_xticks(np.arange(-0.5, 8, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 8, 1), minor=True)
ax.grid(which="minor", color="gray", linestyle='-', linewidth=0.5)
ax.grid(which="major", visible=False)
ax.tick_params(which="minor", bottom=False, left=False)

fig.colorbar(im, ax=ax, label="Inferred Intensity ($\hat{x}$)")
plt.tight_layout()
plt.show()

Saturation harms dye-call. Look at true dye = dye-1 or 5 cases, which are particularly severe.

---

## 6.2 Addressing the Failure Modes

To respect the laws of physics, I must address these failures step-by-step.

### 1. Removing Empty Compartments
First, I eliminate pure background noise from the pipeline entirely. Based on our analysis in Section 2, any compartment with an L2 Norm magnitude below the `noise_threshold` is intrinsically empty and thus excluded from the inverse solver.

### 2. Enforcing Non-Negativity (NNLS)
I replace Ordinary Least Squares with **Non-Negative Least Squares (NNLS)**, which minimizes $||y - Rx||_2^2$ subject to the hard constraint that $x \ge 0$.

In [ ]:
from scipy.optimize import nnls

R_inv = np.linalg.inv(R_norm)

base_intensity = 25000.0

# ==========================================================
# Helper functions
# ==========================================================
def simulate_8_cases_naive(noise_fn, intensity=base_intensity):
    """
    Returns:
        X_true: true 8x8 dye matrix
        X_hat:  naive inverse estimate
        Y_obs:  observed noisy/clipped spectral matrix
    """
    X_true = np.eye(8) * intensity
    Y_clean = R_norm @ X_true
    Y_obs = noise_fn(Y_clean)
    X_hat = R_inv @ Y_obs
    return X_true, X_hat, Y_obs


def simulate_8_cases_nnls(noise_fn, intensity=base_intensity):
    """
    Returns:
        X_true: true 8x8 dye matrix
        X_hat:  NNLS estimate
        Y_obs:  observed noisy/clipped spectral matrix
    """
    X_true = np.eye(8) * intensity
    Y_clean = R_norm @ X_true
    Y_obs = noise_fn(Y_clean)

    X_hat = np.zeros((8, 8))
    for i in range(8):
        X_hat[:, i], _ = nnls(R_norm, Y_obs[:, i])

    return X_true, X_hat, Y_obs


# ==========================================================
# Scenario 1: Static Gaussian noise
# ==========================================================
np.random.seed(42)

mean = 800
sd = 500

def static_noise(Y):
    return np.clip(Y + np.random.normal(mean, sd, Y.shape), 0, 65535)

X_true_static, X_hat_static_naive, Y_static = simulate_8_cases_naive(static_noise)
_, X_hat_static_nnls, _ = simulate_8_cases_nnls(static_noise)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
fig.suptitle("Static Gaussian Noise: Naive Inverse vs NNLS (μ=500, σ=200)", fontsize=14)

# Naive
im1 = axes[0].imshow(X_hat_static_naive, cmap='RdBu_r', vmin=-35000, vmax=35000)
axes[0].set_title("Naive Inverse")
axes[0].set_xlabel("True Dye (Ground Truth)")
axes[0].set_ylabel("Inferred Dye Component")

# NNLS
im2 = axes[1].imshow(X_hat_static_nnls, cmap='Reds', vmin=0, vmax=35000)
axes[1].set_title("NNLS")
axes[1].set_xlabel("True Dye (Ground Truth)")
axes[1].set_ylabel("Inferred Dye Component (>= 0)")

for ax in axes:
    ax.set_xticks(range(8))
    ax.set_yticks(range(8))
    ax.set_xticklabels(range(1, 9))
    ax.set_yticklabels(range(1, 9))
    ax.set_xticks(np.arange(-0.5, 8, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 8, 1), minor=True)
    ax.grid(which="minor", color="gray", linestyle='-', linewidth=0.5)
    ax.grid(which="major", visible=False)
    ax.tick_params(which="minor", bottom=False, left=False)

fig.colorbar(im1, ax=axes[0], label="Inferred Intensity ($\\hat{x}$)")
fig.colorbar(im2, ax=axes[1], label="Inferred Intensity ($\\hat{x}$)")
plt.tight_layout()
plt.show()


# ==========================================================
# Scenario 2: Saturation-only clipping
# ==========================================================
def clipping_noise(Y):
    return np.clip(Y, 0, 65535)

high_intensity = 80000.0

X_true_clip, X_hat_clip_naive, Y_clip = simulate_8_cases_naive(clipping_noise, intensity=high_intensity)
_, X_hat_clip_nnls, _ = simulate_8_cases_nnls(clipping_noise, intensity=high_intensity)

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
fig.suptitle(f"Saturation Only: Naive Inverse vs NNLS (True Intensity={high_intensity:.0f}, Clip=65535)", fontsize=14)

# Naive
im1 = axes[0].imshow(X_hat_clip_naive, cmap='RdBu_r', vmin=-110000, vmax=110000)
axes[0].set_title("Naive Inverse")
axes[0].set_xlabel("True Dye (Ground Truth)")
axes[0].set_ylabel("Inferred Dye Component")

# NNLS
im2 = axes[1].imshow(X_hat_clip_nnls, cmap='Reds', vmin=0, vmax=110000)
axes[1].set_title("NNLS")
axes[1].set_xlabel("True Dye (Ground Truth)")
axes[1].set_ylabel("Inferred Dye Component (>= 0)")

for ax in axes:
    ax.set_xticks(range(8))
    ax.set_yticks(range(8))
    ax.set_xticklabels(range(1, 9))
    ax.set_yticklabels(range(1, 9))
    ax.set_xticks(np.arange(-0.5, 8, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, 8, 1), minor=True)
    ax.grid(which="minor", color="gray", linestyle='-', linewidth=0.5)
    ax.grid(which="major", visible=False)
    ax.tick_params(which="minor", bottom=False, left=False)

fig.colorbar(im1, ax=axes[0], label="Inferred Intensity ($\\hat{x}$)")
fig.colorbar(im2, ax=axes[1], label="Inferred Intensity ($\\hat{x}$)")
plt.tight_layout()
plt.show()

We see NNLS method enhances the dye-call.


----


### 7. Enforcing max 2-dyes and BIC 
To prevent this overfitting, I must enforce our biological constraint: a compartment contains at most 2 dyes. Because our search space is tiny ($\binom{8}{1} + \binom{8}{2} = 36$ valid sub-models), an **exhaustive combinatorial search** is mathematically exact and reasonably fast.

I test all 1-dye and 2-dye combinations using NNLS. To decide between them, I use the **Bayesian Information Criterion (BIC)**. Since a 2-dye model will *always* fit the data better, BIC heavily penalizes the added complexity, ensuring a mixture is only called if it significantly improves the fit over a 1-dye model.

I also apply a physical rule: a secondary dye is rejected if its intensity accounts for <5% of the total, preventing the overfitting of trace spectral bleed-through.



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import itertools
from scipy.optimize import nnls

# ==========================================================
# Exhaustive model evaluation with BIC
# ==========================================================
def evaluate_all_36_models_bic(y_obs, R_basis, min_fraction=0.05):
    n_dyes = R_basis.shape[1]
    n_channels = len(y_obs)
    results = []

    # 1-dye models
    for i in range(n_dyes):
        R_sub = R_basis[:, [i]]
        x_hat, _ = nnls(R_sub, y_obs)
        y_hat = R_sub @ x_hat
        rss = max(np.sum((y_obs - y_hat) ** 2), 1e-10)
        bic = n_channels * np.log(rss / n_channels) + 1 * np.log(n_channels)

        results.append({
            'model': f"Dye {i+1}",
            'bic': bic,
            'y_hat': y_hat,
            'x_hat': x_hat
        })

    # 2-dye models
    for pair in itertools.combinations(range(n_dyes), 2):
        R_sub = R_basis[:, list(pair)]
        x_hat, _ = nnls(R_sub, y_obs)
        y_hat = R_sub @ x_hat

        total_x = np.sum(x_hat)

        if total_x > 0 and not np.any((x_hat / total_x) < min_fraction):
            rss = max(np.sum((y_obs - y_hat) ** 2), 1e-10)
            bic = n_channels * np.log(rss / n_channels) + 2 * np.log(n_channels)
        else:
            bic = np.nan

        results.append({
            'model': f"Dye {pair[0]+1} + {pair[1]+1}",
            'bic': bic,
            'y_hat': y_hat,
            'x_hat': x_hat
        })

    df = pd.DataFrame(results)
    return df.sort_values('bic', na_position='last').reset_index(drop=True)
# ==========================================================
# Synthetic noise model
# ==========================================================
def synthetic_noise(y, mean=300, sd=400):
    noisy = y + np.random.normal(mean, sd, size=y.shape)
    return np.clip(noisy, 0, 65535)


# ==========================================================
# Build synthetic test cases
# ==========================================================
np.random.seed(42)

# Synthetic 2-dye
x_2dye = np.zeros(8)
x_2dye[0] = 15000
x_2dye[3] = 10000
y_syn2_sample = synthetic_noise(R_norm @ x_2dye)

# Synthetic 1-dye
x_1dye = np.zeros(8)
x_1dye[5] = 18000
y_syn1_sample = synthetic_noise(R_norm @ x_1dye)


# ==========================================================
# Real examples
# ==========================================================
if 'Y_real' in globals() and len(Y_real) > 0:
    
    # previously sampled real signal
    if 'magnitudes' in globals() and 'noise_threshold' in globals():
        real_sig_idx = np.where(magnitudes > noise_threshold)[0]
        sample_real_idx = real_sig_idx[42] if len(real_sig_idx) > 42 else 0
    else:
        sample_real_idx = 0

    y_real_sample = Y_real[sample_real_idx]

    # additional fixed real row
    y_real_2900 = Y_real[2900] if len(Y_real) > 2900 else Y_real[-1]

else:
    sample_real_idx = 0
    y_real_sample = np.zeros(8)
    y_real_2900 = np.zeros(8)


# ==========================================================
# Assemble test cases
# ==========================================================
test_cases = [
    {"y": y_syn2_sample, "title": "Synthetic Data (True: Dyes 1 & 4)"},
    {"y": y_syn1_sample, "title": "Synthetic Data (True: Dye 6)"},
    {"y": y_real_sample, "title": f"Real Data (Compartment {sample_real_idx})"},
    {"y": y_real_2900, "title": "Real Data (Compartment 2900)"}
]


# ==========================================================
# Plot results
# ==========================================================
fig, axes = plt.subplots(
    4, 2,
    figsize=(11, 13),
    gridspec_kw={'width_ratios': [2, 1]}
)

fig.suptitle("Exhaustive Search Results (Lowest 10 BIC Scores)", fontsize=16)

for i, case in enumerate(test_cases):

    y_obs = case["y"]
    df_res = evaluate_all_36_models_bic(y_obs, R_norm)

    valid_df = df_res.dropna(subset=['bic']).reset_index(drop=True)
    plot_df = valid_df.head(10)

    # --------------------------------
    # BIC ranking
    # --------------------------------
    ax_bic = axes[i, 0]
    colors = ['crimson'] + ['lightsteelblue'] * (len(plot_df) - 1)

    ax_bic.bar(
        range(len(plot_df)),
        plot_df['bic'],
        color=colors,
        edgecolor='black',
        linewidth=0.5
    )

    ax_bic.set_xticks(range(len(plot_df)))
    ax_bic.set_xticklabels(plot_df['model'], rotation=45, ha='right')
    ax_bic.set_title(f"{case['title']} - Lowest 10 BIC Scores")
    ax_bic.set_ylabel("BIC (Lower is better)")

    # --------------------------------
    # Spectral fit
    # --------------------------------
    ax_fit = axes[i, 1]

    best_model = valid_df.iloc[0]

    ax_fit.plot(range(1, 9), y_obs, 'ko-', label='Observed', markersize=6)
    ax_fit.plot(
        range(1, 9),
        best_model['y_hat'],
        'rX--',
        label=f'Best Fit ({best_model["model"]})',
        markersize=8
    )

    ax_fit.set_title(f"Best Model: {best_model['model']}")
    ax_fit.set_xticks(range(1, 9))
    ax_fit.set_xlabel("Detection Channel")
    ax_fit.set_ylabel("Intensity")
    ax_fit.legend()

plt.tight_layout()
plt.subplots_adjust(top=0.95, hspace=0.5)
plt.show()



### 8. Goodness-of-Fit ($R^2$) Safeguard
Finally, computing $R^2$ between the observed $y$ and predicted $\hat{y}$ acts as a crucial safeguard against the anomalies we identified earlier (sensor saturation at 65535, dust, or 3+ dyes). A low $R^2$ flags the compartment as "Low Confidence", safely quarantining bad data.

I will demonstrate how R2 values can make a difference below and define the function.

In [ ]:
# ==========================================================
# R^2 scoring and family comparison plots
# (assumes evaluate_all_36_models_bic(...) is already defined above)
# ==========================================================

def compute_r2(y_obs, y_hat):
    ss_res = np.sum((y_obs - y_hat) ** 2)
    ss_tot = np.sum((y_obs - np.mean(y_obs)) ** 2)
    if ss_tot <= 1e-12:
        return np.nan
    return 1 - ss_res / ss_tot


def evaluate_r2(df_bic, y_obs, r2_thresh=0.85):
    """
    Add R^2 to each candidate model and return:
      - best overall model among those with R^2 >= r2_thresh
      - full scored dataframe sorted by BIC
    """
    df = df_bic.copy()
    df['r2'] = df['y_hat'].apply(lambda y_hat: compute_r2(y_obs, y_hat))

    valid = df.dropna(subset=['bic', 'r2']).copy()

    passed_df = valid[valid['r2'] >= r2_thresh]
    if passed_df.empty:
        best_overall = None
    else:
        best_overall = passed_df.sort_values(['bic', 'r2'], ascending=[True, False]).iloc[0]

    return best_overall, valid.sort_values('bic').reset_index(drop=True)


# ==========================================================
# Family helpers
# ==========================================================
def pick_best_family(df_scored, family_size):
    is_two = df_scored['model'].str.contains(r'\+', regex=True)
    family_df = df_scored[is_two] if family_size == 2 else df_scored[~is_two]
    family_df = family_df.dropna(subset=['r2', 'bic'])

    if family_df.empty:
        return None

    return family_df.sort_values(['bic', 'r2'], ascending=[True, False]).iloc[0]


def build_plot_df_with_family_hits(df_scored, top_n=10):
    """
    Start from top_n by BIC, then force-include the best 1-dye and best 2-dye
    if they are not already present. Re-sort by BIC afterward.
    """
    top_df = df_scored.head(top_n).copy()

    best_1d = pick_best_family(df_scored, 1)
    best_2d = pick_best_family(df_scored, 2)

    extras = []
    for best in [best_1d, best_2d]:
        if best is not None and best['model'] not in top_df['model'].values:
            extras.append(best.to_dict())

    if extras:
        top_df = pd.concat([top_df, pd.DataFrame(extras)], ignore_index=True)
        top_df = top_df.drop_duplicates(subset='model')
        top_df = top_df.sort_values('bic').reset_index(drop=True)

    return top_df, best_1d, best_2d


# ==========================================================
# Build same exact data as before: 2 synthetic + 2 real
# Reuse synthetic_noise(...) from above if already defined
# ==========================================================
np.random.seed(42)

# Synthetic 2-dye
x_2dye = np.zeros(8)
x_2dye[0] = 15000
x_2dye[3] = 10000
y_syn2_sample = synthetic_noise(R_norm @ x_2dye)

# Synthetic 1-dye
x_1dye = np.zeros(8)
x_1dye[5] = 18000
y_syn1_sample = synthetic_noise(R_norm @ x_1dye)

# Real data examples
if 'Y_real' in globals() and len(Y_real) > 0:
    if 'magnitudes' in globals() and 'noise_threshold' in globals():
        real_sig_idx = np.where(magnitudes > noise_threshold)[0]
        sample_real_idx = real_sig_idx[42] if len(real_sig_idx) > 42 else 0
    else:
        sample_real_idx = 0

    y_real_sample = Y_real[sample_real_idx]
    y_real_2900 = Y_real[2900] if len(Y_real) > 2900 else Y_real[-1]
else:
    sample_real_idx = 0
    y_real_sample = np.zeros(8)
    y_real_2900 = np.zeros(8)

test_cases = [
    {"y": y_syn2_sample, "title": "Synthetic Data (True: Dyes 1 & 4)"},
    {"y": y_syn1_sample, "title": "Synthetic Data (True: Dye 6)"},
    {"y": y_real_sample, "title": f"Real Data (Compartment {sample_real_idx})"},
    {"y": y_real_2900, "title": "Real Data (Compartment 2900)"}
]


# ==========================================================
# Plot layout: 4 rows x 2 cols
# ==========================================================
from matplotlib.patches import Patch

fig, axes = plt.subplots(
    4, 2,
    figsize=(11, 13),
    gridspec_kw={'width_ratios': [2, 1]}
)

fig.suptitle(
    "Exhaustive Search Results (Top BICs + Best 1-Dye vs 2-Dye Fits)",
    fontsize=16,
    y=0.985
)


for i, case in enumerate(test_cases):
    y_obs = case["y"]

    df_bic = evaluate_all_36_models_bic(y_obs, R_norm)
    best_overall, df_scored = evaluate_r2(df_bic, y_obs, r2_thresh=0.85)

    plot_df, best_1d, best_2d = build_plot_df_with_family_hits(df_scored, top_n=10)

    # --------------------------------------------------
    # Left panel: BIC ranking
    # --------------------------------------------------
    ax_bic = axes[i, 0]

    colors = []
    for _, row in plot_df.iterrows():
        if best_1d is not None and row['model'] == best_1d['model']:
            colors.append('darkorange')
        elif best_2d is not None and row['model'] == best_2d['model']:
            colors.append('teal')
        else:
            colors.append('lightsteelblue')

    ax_bic.bar(
        range(len(plot_df)),
        plot_df['bic'],
        color=colors,
        edgecolor='black',
        linewidth=0.5
    )

    ax_bic.set_xticks(range(len(plot_df)))
    ax_bic.set_xticklabels(plot_df['model'], rotation=45, ha='right')
    ax_bic.set_title(f"{case['title']} - Top BIC Models")
    ax_bic.set_ylabel("BIC (Lower is better)")

    legend_handles = [
        Patch(facecolor='darkorange', edgecolor='black', label='Best 1-dye BIC'),
        Patch(facecolor='teal', edgecolor='black', label='Best 2-dye BIC'),
        Patch(facecolor='lightsteelblue', edgecolor='black', label='Other top models'),
    ]
    ax_bic.legend(handles=legend_handles, loc='right', fontsize=8, framealpha=0.9)

    # --------------------------------------------------
    # Right panel: observed + best 1-dye + best 2-dye
    # --------------------------------------------------
    ax_fit = axes[i, 1]
    x_axis = np.arange(1, len(y_obs) + 1)

    ax_fit.plot(
        x_axis, y_obs, 'ko-',
        label='Observed',
        markersize=6,
        alpha=0.8
    )

    if best_1d is not None:
        ax_fit.plot(
            x_axis,
            best_1d['y_hat'],
            ':o',
            color='darkorange',
            linewidth=2.5,
            markersize=6,
            label=(
                f"1-dye fit ({best_1d['model']})\n"
                f"R² = {best_1d['r2']:.3f}, BIC = {best_1d['bic']:.1f}"
            )
        )

    if best_2d is not None:
        ax_fit.plot(
            x_axis,
            best_2d['y_hat'],
            '--s',
            color='teal',
            linewidth=2.5,
            markersize=6,
            label=(
                f"2-dye fit ({best_2d['model']})\n"
                f"R² = {best_2d['r2']:.3f}, BIC = {best_2d['bic']:.1f}"
            )
        )

    # Title with delta summaries
    title_lines = [case['title']]
    if best_1d is not None and best_2d is not None:
        delta_r2 = best_2d['r2'] - best_1d['r2']
        delta_bic = best_2d['bic'] - best_1d['bic']
        # title_lines.append(f"ΔR² (2D-1D) = {delta_r2:+.3f}, ΔBIC (2D-1D) = {delta_bic:+.1f}")
    elif best_1d is not None:
        title_lines.append("Only 1-dye fit available")
    elif best_2d is not None:
        title_lines.append("Only 2-dye fit available")

    ax_fit.set_title("\n".join(title_lines))
    ax_fit.set_xticks(x_axis)
    ax_fit.set_xlabel("Detection Channel")
    ax_fit.set_ylabel("Intensity")

    ax_fit.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=8, framealpha=0.9)

plt.tight_layout(rect=[0, 0, 0.92, 0.955])
plt.subplots_adjust(top=0.93, hspace=0.6)
plt.show()

We observe that in choosing between 1-dye and 2-dye best fits, lower BIC and higher R2 go together.

The above visualizations clearly confirm that our synthetic 2-dye edge cases are correctly handled and validated by the logic sequence. 

Additionally, let's explore how integrating a secondary 2-dye allowance improves our goodness-of-fit distributions natively. We can compare the distribution of the $R^2$ scores for a strict 1-dye constraint to those of an adaptive 1-to-2 dye environment.

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ====================================================================
# Visualizing: The R^2 Improvement using 2-Dye Models
# ====================================================================

r2_1dye_list = []
r2_overall_list = []

# Use only signal compartments that pass the noise threshold
signal_indices = np.where(magnitudes > noise_threshold)[0]

for idx in signal_indices:
    y_obs = Y_real[idx]

    # Evaluate all 1- and 2-dye models and compute R^2 for each
    df_bic = evaluate_all_36_models_bic(y_obs, R_norm)
    _, df_scored = evaluate_r2(df_bic, y_obs, r2_thresh=0.0)

    # Best R^2 if restricted to 1-dye models only
    df_1dye = df_scored[~df_scored['model'].str.contains(r'\+', regex=True)]
    max_r2_1dye = df_1dye['r2'].max() if not df_1dye['r2'].isna().all() else 0.0
    r2_1dye_list.append(max_r2_1dye)

    # Best R^2 allowing both 1- and 2-dye mixtures
    max_r2_overall = df_scored['r2'].max() if not df_scored['r2'].isna().all() else 0.0
    r2_overall_list.append(max_r2_overall)

# Figure with a "broken" y-axis: top 6000–9000, bottom 0–150
fig, (ax_high, ax_low) = plt.subplots(
    2, 1, figsize=(7, 6), sharex=True,
    gridspec_kw={'height_ratios': [1, 1], 'hspace': 0.05}
)

bins = np.linspace(0.4, 1.0, 60)

# Plot histograms separately so the color/category mapping is unambiguous
ax_high.hist(
    r2_1dye_list,
    bins=bins,
    color='darkorange',
    edgecolor='black',
    linewidth=0.5,
    alpha=0.85
)
ax_high.hist(
    r2_overall_list,
    bins=bins,
    color='teal',
    edgecolor='black',
    linewidth=0.5,
    alpha=0.55
)

ax_low.hist(
    r2_1dye_list,
    bins=bins,
    color='darkorange',
    edgecolor='black',
    linewidth=0.5,
    alpha=0.85
)
ax_low.hist(
    r2_overall_list,
    bins=bins,
    color='teal',
    edgecolor='black',
    linewidth=0.5,
    alpha=0.55
)

# Threshold line on both
for ax in (ax_high, ax_low):
    ax.axvline(0.85, color='crimson', linestyle='--', linewidth=2)

# Y-limits for the two ranges
ax_high.set_ylim(6000, 9000)
ax_low.set_ylim(0, 150)

ax_high.set_ylabel('Number of Compartments', fontsize=12)
ax_low.set_ylabel('Zoom (0–150)', fontsize=10)
ax_low.set_xlabel('Goodness-of-Fit ($R^2$ Score)', fontsize=12)

# Count how many were saved by the 2-dye model
saved_count = sum(
    1 for r1, r2 in zip(r2_1dye_list, r2_overall_list)
    if r1 < 0.85 and r2 >= 0.85
)

ax_high.set_title(
    'Distribution of Quality of Fit ($R^2$) Across All Signal Compartments\n'
    f'({saved_count} compartments rescued by modeling a 2-dye mixture)',
    fontsize=14
)

# Explicit legend handles so labels match the actual visual elements
legend_handles = [
    Patch(facecolor='darkorange', edgecolor='black', linewidth=0.5, alpha=0.85,
          label='Best $R^2$ (Restricted to 1-Dye)'),
    Patch(facecolor='teal', edgecolor='black', linewidth=0.5, alpha=0.85,
          label='Best $R^2$ (Allowing 2-Dye Mixtures)'),
    Line2D([0], [0], color='crimson', linestyle='--', linewidth=2,
           label='Physical Quality Threshold (0.85)')
]

ax_high.legend(handles=legend_handles, loc='upper left', fontsize=11)

# Hide spines between high and low axes to emphasize the "break"
ax_high.spines['bottom'].set_visible(False)
ax_low.spines['top'].set_visible(False)
ax_high.tick_params(labelbottom=False)

# Draw diagonal lines to indicate the axis break
kwargs = dict(transform=ax_high.transAxes, color='k', clip_on=False)
ax_high.plot((-0.015, +0.015), (-0.01, +0.01), **kwargs)
ax_high.plot((0.985, 1.015), (-0.01, +0.01), **kwargs)

kwargs = dict(transform=ax_low.transAxes, color='k', clip_on=False)
ax_low.plot((-0.015, +0.015), (1 - 0.01, 1 + 0.01), **kwargs)
ax_low.plot((0.985, 1.015), (1 - 0.01, 1 + 0.01), **kwargs)

ax_high.set_xlim(0.4, 1.05)
plt.tight_layout()
plt.show()

We see that with allowing for 2-dye mixtures, the R2 value enhances.

With empty compartments flagged and poorly fitted arrays discarded by the $R^2$ logic filter, let's tally the model adjustments and view the distribution of corrected assignments across the whole dataset.

In [ ]:
def process_entire_dataset(Y_obs, R_basis, dist_thresh, r2_thresh=0.85):
    """Processes the dataset through the sequential decision funnel."""
    predictions = []
    magnitudes = np.linalg.norm(Y_obs, axis=1)
    
    for i in range(Y_obs.shape[0]):
        y_obs = Y_obs[i]
        
        # GATE 1: Empty Compartments
        if magnitudes[i] < dist_thresh:
            predictions.append({'Compartment': i, 'Status_Pre_R2': 'Background Noise', 'Status_Post_R2': 'Background Noise', 'Dyes_Present': []})
            continue
            
        # GATE 2: L0-Constrained BIC Selection
        df_bic = evaluate_all_36_models_bic(y_obs, R_basis)
        
        # Determine Pre-R2 purely based on the top BIC physical model
        best_bic_only = df_bic.dropna(subset=['bic']).sort_values('bic').iloc[0]
        status_pre = '2-Dye Mixture' if '+' in str(best_bic_only['model']) else '1-Dye Signal'
            
        # GATE 3: R^2 Anomaly Check via passing output to evaluate_r2
        best_r2_model, _ = evaluate_r2(df_bic, y_obs, r2_thresh)
        
        if best_r2_model is None:
            status_post = 'Anomaly (Low R2)'
            dyes_1_indexed = []
        else:
            model_str = str(best_r2_model['model'])
            if '+' in model_str:
                status_post = '2-Dye Mixture'
                dyes_1_indexed = [int(x) for x in model_str.replace("Dye ", "").split(" + ")]
            else:
                status_post = '1-Dye Signal'
                dyes_1_indexed = [int(model_str.replace("Dye ", ""))]
            
        predictions.append({'Compartment': i, 'Status_Pre_R2': status_pre, 'Status_Post_R2': status_post, 'Dyes_Present': dyes_1_indexed})
        
    return pd.DataFrame(predictions)

print("Executing pipeline on real dataset (Evaluating all 36 combinations for ~10,000 components..)")
df_results = process_entire_dataset(Y_real, R_norm, dist_thresh=noise_threshold)
df_results['Status'] = df_results['Status_Post_R2']  # Compatibility alias for the final report

# Visualize how the results change (Counts) pre and post validation
pre_counts = df_results['Status_Pre_R2'].value_counts()
post_counts = df_results['Status_Post_R2'].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
categories = ['1-Dye Signal', '2-Dye Mixture', 'Anomaly (Low R2)']
pre_vals = [pre_counts.get('1-Dye Signal', 0), pre_counts.get('2-Dye Mixture', 0), pre_counts.get('Anomaly (Low R2)', 0)]
post_vals = [post_counts.get('1-Dye Signal', 0), post_counts.get('2-Dye Mixture', 0), post_counts.get('Anomaly (Low R2)', 0)]

x = np.arange(len(categories))
width = 0.35
ax.bar(x - width/2, pre_vals, width, label='Pre-R$^2$ Filter (BIC Only)', color='lightsteelblue')
ax.bar(x + width/2, post_vals, width, label='Post-R$^2$ Filter (Final)', color='crimson')

ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylabel("Number of Compartments")
ax.set_title("Population Shifts: Impact of Filtering by R$^2$ $\geq$ 0.85", fontsize=14)
ax.legend()

# Annotate counts inside/above bars
for i in range(len(categories)):
    ax.text(x[i] - width/2, pre_vals[i] + max(max(pre_vals), max(post_vals))*0.01, str(pre_vals[i]), ha='center', va='bottom', fontsize=10)
    ax.text(x[i] + width/2, post_vals[i] + max(max(pre_vals), max(post_vals))*0.01, str(post_vals[i]), ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

I see that R2-filter did not in fact change anything. It was not needed. But I'll keep it as a safety net.

Almost done. Let's package up the results of this assignment inside a visual terminal readout. The outputs map specifically to the project specs:

- How many times the algorithm identified 0-, 1-, and 2-dye compartments.
- Confidence intervals of the models.
- The top requested combinations of distinct 2-dye superpositions.


---

## 9. The Complete Algorithm & Execution

I will now implement this logic functionally and execute it across the entire 10,000-compartment dataset. The pipeline follows a sequential decision funnel:
1. Gate out Empty Compartments using the dynamically calculated `noise_threshold`.
2. Find the optimal constrained layout using BIC and NNLS.
3. Validate and flag physical anomalies using $R^2$.

---

## 10. Final Report

The structured output detailing the final assignments for the dataset is generated below.

In [ ]:
# ====================================================================
# FINAL REPORTING & VISUALIZATIONS (robust to Status column naming)
# ====================================================================

print("Generating final classification report...")
df_scored = process_entire_dataset(
    Y_obs=Y_real,
    R_basis=R_raw,
    dist_thresh=noise_threshold,
    r2_thresh=0.85
)

status_col = 'Status_Post_R2' if 'Status_Post_R2' in df_scored.columns else 'Status'
if status_col not in df_scored.columns:
    raise KeyError("Expected one of ['Status_Post_R2', 'Status'] in dataframe columns")

# Split full vs non-background once, then reuse consistently
df_res = df_scored[df_scored[status_col] != 'Background Noise'].copy()

total = len(df_scored)
n_empty = (df_scored[status_col] == 'Background Noise').sum()

print("=" * 60)
print(f"Total Compartments        : {total}")
print(f"Empty Compartments        : {n_empty} ({n_empty / max(total, 1) * 100:.1f}%)")
print(f"Data Compartments Analysed: {len(df_res)}")
print("=" * 60)

def get_single_dye(dyes_list):
    return f"Dye {dyes_list[0]}" if len(dyes_list) == 1 else None

def get_top_2_dyes(dyes_list):
    return tuple(sorted(dyes_list)) if len(dyes_list) == 2 else None

def format_dye_pair(pair):
    return "" if pair is None else f"Dye {pair[0]} + {pair[1]}"

# 1-dye summary
df_1dye_only = df_res[df_res[status_col] == '1-Dye Signal'].copy()
df_1dye_only['Single_Dye'] = df_1dye_only['Dyes_Present'].apply(get_single_dye)
dye1_counts = df_1dye_only['Single_Dye'].value_counts().sort_index()

print("\n[ 1-Dye Distribution ]")
print("  Dye     : Count")
print("-" * 25)
for dye, cnt in dye1_counts.items():
    print(f"  {dye:7s} : {cnt:>5} ({cnt / max(len(df_res), 1) * 100:>5.1f}%)")
print("=" * 60)

# Final class distribution
counts = df_res[status_col].value_counts()
print("\n[ Final Dye Classifications ]")
for cat, cnt in counts.items():
    print(f"  {cat:15s} : {cnt:>5} ({cnt / max(len(df_res), 1) * 100:>5.1f}%)")

anomalies_count = (df_res[status_col] == 'Anomaly (Low R2)').sum()
print(f"  --> Specifically Rejected as Anomalies: {anomalies_count} ({anomalies_count / max(len(df_res), 1) * 100:.1f}%)")
print("=" * 60)

# All 2-dye combinations (no head limit)
df_2dye = df_res[df_res[status_col] == '2-Dye Mixture'].copy()
df_2dye['Top_2_Indices'] = df_2dye['Dyes_Present'].apply(get_top_2_dyes)
all_combos = df_2dye['Top_2_Indices'].value_counts()

print(f"\n[ All 2-Dye Combinations ({len(all_combos)} unique pairs) ]")
print("  Combination           : Count")
print("-" * 40)
for combo, cnt in all_combos.items():
    if combo is not None:
        print(f"  {format_dye_pair(combo):22s}: {cnt:>5}")
print("=" * 60)

# ====================================================================
# Combined dye occurrence counts (1-dye + contributions from 2-dye)
# ====================================================================
combined_counts = dye1_counts.copy().reindex(
    [f"Dye {i}" for i in range(1, 9)], fill_value=0
)

for combo, cnt in all_combos.items():
    if combo is not None:
        for dye_idx in combo:
            label = f"Dye {dye_idx}"
            if label in combined_counts.index:
                combined_counts[label] += cnt
            else:
                combined_counts[label] = cnt

combined_counts = combined_counts.sort_index()

print("\n[ FINAL DYE OCCURRENCE COUNTS (1-Dye + 2-Dye Combined) ]")
print("  Dye     : Total   (1-Dye only  +  2-Dye contributions)")
print("-" * 55)
for dye_label in combined_counts.index:
    total_cnt = combined_counts[dye_label]
    only1 = dye1_counts.get(dye_label, 0)
    from_2 = total_cnt - only1
    print(f"  {dye_label:7s} : {total_cnt:>5}   ({only1} from 1-dye  +  {from_2} from 2-dye)")
print("=" * 60)

# ====================================================================
# SUMMARY VISUALIZATIONS
# ====================================================================
import seaborn as sns
sns.set_theme(style="whitegrid")

# --- Figure 1: Overall classification + 1-dye distribution ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

overall_counts = df_scored[status_col].value_counts()
overall_labels = overall_counts.index
overall_vals = overall_counts.values

ax1.bar(overall_labels, overall_vals, color='lightsteelblue', edgecolor='black', alpha=0.85)
ax1.set_title("Overall Data Classification", fontsize=13, fontweight='bold')
ax1.set_ylabel("Compartment Count")
ax1.set_xticks(range(len(overall_labels)))
ax1.set_xticklabels(overall_labels, rotation=25, ha='right')
for i, v in enumerate(overall_vals):
    ax1.text(i, v + (max(overall_vals) * 0.01 if len(overall_vals) else 0), str(v), ha='center', fontweight='bold')

ax2.bar(dye1_counts.index, dye1_counts.values, color='coral', edgecolor='black', alpha=0.85)
ax2.set_title(f"1-Dye Specific Call Distribution ({dye1_counts.sum()} total)", fontsize=13, fontweight='bold')
ax2.set_ylabel("Count")
ax2.set_xticks(range(len(dye1_counts.index)))
ax2.set_xticklabels(dye1_counts.index, rotation=45, ha='right')
if len(dye1_counts.values):
    for i, v in enumerate(dye1_counts.values):
        ax2.text(i, v + (max(dye1_counts.values) * 0.02), str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.subplots_adjust(bottom=0.2)
plt.show()

# --- Figure 2: All 2-dye combinations (separate plot) ---
valid_combos = all_combos[all_combos.index.notnull()]
combo_labels = [format_dye_pair(c) for c in valid_combos.index]
combo_values = valid_combos.values

fig2, ax_combo = plt.subplots(figsize=(max(10, len(combo_labels) * 0.55), 6))
ax_combo.bar(combo_labels, combo_values, color='mediumseagreen', edgecolor='black', alpha=0.85)
ax_combo.set_title(f"All 2-Dye Mixture Combinations ({len(valid_combos)} unique pairs, {int(combo_values.sum()*2)} total dyes)", fontsize=13, fontweight='bold')
ax_combo.set_xlabel("Dye Pair")
ax_combo.set_ylabel("Occurrence Count")
ax_combo.set_xticks(range(len(combo_labels)))
ax_combo.set_xticklabels(combo_labels, rotation=45, ha='right')
if len(combo_values):
    for i, v in enumerate(combo_values):
        ax_combo.text(i, v + (max(combo_values) * 0.01), str(v), ha='center', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Figure 3: Final combined dye counts (stacked bar) ---
fig3, ax_final = plt.subplots(figsize=(10, 6))

dye_labels_final = combined_counts.index.tolist()
vals_1dye = [dye1_counts.get(d, 0) for d in dye_labels_final]
vals_2dye = [combined_counts[d] - dye1_counts.get(d, 0) for d in dye_labels_final]
x_pos = range(len(dye_labels_final))

ax_final.bar(x_pos, vals_1dye, label='From 1-Dye results', color='coral', edgecolor='black', alpha=0.85)
ax_final.bar(x_pos, vals_2dye, bottom=vals_1dye, label='From 2-Dye results', color='mediumseagreen', edgecolor='black', alpha=0.85)

# Mark the 1-dye-only subtotal with a horizontal tick line on each bar
for i, v1 in enumerate(vals_1dye):
    ax_final.plot([i - 0.4, i + 0.4], [v1, v1], color='black', linewidth=2, linestyle='--', zorder=5)
    ax_final.text(i, v1 - (max(combined_counts.values) * 0.035), f'{v1}', ha='center', va='top',
                  fontweight='bold', fontsize=11,
                  )

ax_final.set_title(f" FINAL DYE OCCURRENCE COUNTS   ({int(combined_counts.sum())} total dye occurrences)\n(1-Dye Signal + 2-Dye Mixture contributions combined)", fontsize=13, fontweight='bold')
ax_final.set_xlabel("Dye")
ax_final.set_ylabel("Total Occurrence Count")
ax_final.set_xticks(x_pos)
ax_final.set_xticklabels(dye_labels_final, rotation=0)

from matplotlib.lines import Line2D
# legend_handles_extra = [Line2D([0], [0], color='black', linewidth=2, linestyle='--', label='1-Dye only subtotal')]
ax_final.legend(handles=ax_final.containers[:2]) # + legend_handles_extra)

for i, total_v in enumerate(combined_counts.values):
    ax_final.text(i, total_v + (max(combined_counts.values) * 0.01), str(total_v), ha='center', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()


## 11. Diagnostic Analysis: Empty vs Signal

I revisit the "empty compartment" signals to investigate how we can distinguish "weak" signals to "background (empty)"

For each compartment I compute:
- An unconstrained NNLS fit using all dyes, recording how many dyes carry ≥5% of the inferred intensity and the resulting R².
- A BIC-selected 1- or 2-dye model, recording its R² and the BIC gap to the runner-up model.

These summaries show whether empty compartments structurally differ from real signals and whether BIC plus the magnitude gate cleanly separates noise from true dye mixtures.

<div style="padding: 14px 16px; border-left: 8px solid #da18d3; border-radius: 8px; font-size: 1.15em; line-height: 1.4;">
    <strong style="font-size: 1.25em;">Important:</strong><br/>
    <strong>In particular, R² acts as a natural confidence metric: if the best-fit model's R² falls below a chosen threshold, that compartment should be flagged and categorized separately instead of being forced into a standard signal class, so we can know empty is empty and not signal.</strong>
</div>

Thus showing that the "empty-compartments" whose L2 norm magnitude is below the threshold probably truly are "empty".

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import nnls

# ==========================================================
# Correct BIC model evaluation
# ==========================================================
def generate_models(n_dyes):
    models = [[i] for i in range(n_dyes)]
    models += [list(pair) for pair in itertools.combinations(range(n_dyes), 2)]
    return models


def bic_score(rss, n, k):
    rss = max(float(rss), 1e-12)  # avoid log(0)
    return n * np.log(rss / n) + k * np.log(n)


def compute_r2(y_obs, y_hat):
    ss_res = np.sum((y_obs - y_hat) ** 2)
    ss_tot = np.sum((y_obs - np.mean(y_obs)) ** 2)
    if ss_tot <= 1e-12:
        return np.nan
    return 1 - ss_res / ss_tot


def evaluate_all_36_models_bic(y_obs, R_basis, min_fraction=0.05):
    """
    Evaluate all 1-dye and 2-dye NNLS models.

    Returns
    -------
    best_model : list of dye indices (0-based)
    best_r2 : R^2 of best-BIC model
    best_family : 1 or 2
    delta_bic : runnerup_bic - best_bic
    best_xhat : inferred dye weights for winning model
    best_yhat : fitted spectrum for winning model
    df_models : full model table
    """
    n_channels, n_dyes = R_basis.shape
    models = generate_models(n_dyes)

    rows = []

    for m in models:
        R_sub = R_basis[:, m]   # CORRECT: select dye columns
        x_hat, _ = nnls(R_sub, y_obs)
        y_hat = R_sub @ x_hat
        rss = np.sum((y_obs - y_hat) ** 2)
        r2 = compute_r2(y_obs, y_hat)

        # Optional: for 2-dye models reject tiny secondary components
        if len(m) == 2:
            total_x = np.sum(x_hat)
            if total_x > 0:
                frac = x_hat / total_x
                if np.any(frac < min_fraction):
                    bic = np.nan
                else:
                    bic = bic_score(rss, len(y_obs), len(m))
            else:
                bic = np.nan
        else:
            bic = bic_score(rss, len(y_obs), len(m))

        rows.append({
            'model': m,
            'family': len(m),
            'rss': rss,
            'r2': r2,
            'bic': bic,
            'x_hat': x_hat,
            'y_hat': y_hat
        })

    df_models = pd.DataFrame(rows)
    valid = df_models.dropna(subset=['bic']).sort_values('bic').reset_index(drop=True)

    if len(valid) == 0:
        return None, np.nan, np.nan, np.nan, None, None, df_models

    best = valid.iloc[0]
    second_bic = valid.iloc[1]['bic'] if len(valid) > 1 else np.nan
    delta_bic = second_bic - best['bic'] if pd.notna(second_bic) else np.nan

    return (
        best['model'],
        best['r2'],
        best['family'],
        delta_bic,
        best['x_hat'],
        best['y_hat'],
        df_models
    )


# ==========================================================
# Build diagnostic dataframe for empty vs signal
# IMPORTANT: evaluate BIC for ALL compartments
# ==========================================================
df_empty_compare_rows = []

magnitudes = np.linalg.norm(Y_real, axis=1)

for i, y in enumerate(Y_real):
    mag = magnitudes[i]
    group = 'Empty' if mag < noise_threshold else 'Signal'

    # Dense / unconstrained NNLS using all dyes
    x_unconstrained, _ = nnls(R_raw, y)   # R_raw is (8 channels x 8 dyes)
    y_unconstrained = R_raw @ x_unconstrained

    r2_unconstrained = compute_r2(y, y_unconstrained)

    total_x = np.sum(x_unconstrained)
    if total_x > 0:
        num_dyes_unconstrained = np.sum((x_unconstrained / total_x) >= 0.05)
    else:
        num_dyes_unconstrained = 0

    # BIC model selection for ALL compartments
    best_model, r2_bic, family, delta_bic, best_xhat, best_yhat, df_models = \
        evaluate_all_36_models_bic(y, R_raw, min_fraction=0.05)

    df_empty_compare_rows.append({
        'Compartment': i,
        'Group': group,
        'Magnitude': mag,
        'Unconstrained_NumDyes': num_dyes_unconstrained,
        'Unconstrained_R2': r2_unconstrained,
        'BIC_BestFamily': family,
        'BIC_R2': r2_bic,
        'BIC_DeltaToSecond': delta_bic
    })

df_empty_compare = pd.DataFrame(df_empty_compare_rows)

empty_df = df_empty_compare[df_empty_compare['Group'] == 'Empty'].copy()
signal_df = df_empty_compare[df_empty_compare['Group'] == 'Signal'].copy()

print("Counts")
print(df_empty_compare['Group'].value_counts())
print("\nNon-null BIC fields:")
print(df_empty_compare[['BIC_BestFamily', 'BIC_R2', 'BIC_DeltaToSecond']].notna().sum())

print("\nEmpty BIC family counts:")
print(empty_df['BIC_BestFamily'].value_counts(dropna=False).sort_index())

print("\nSignal BIC family counts:")
print(signal_df['BIC_BestFamily'].value_counts(dropna=False).sort_index())


# ==========================================================
# Visualization
# ==========================================================
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

n_empty = len(empty_df)
n_signal = len(signal_df)
w = 0.38

# --------------------------------------------------
# 1. Magnitude distributions
# --------------------------------------------------
for group, color in [('Empty', 'lightcoral'), ('Signal', 'steelblue')]:
    vals = df_empty_compare.loc[df_empty_compare['Group'] == group, 'Magnitude']
    axes[0, 0].hist(
        vals,
        bins=40,
        alpha=0.65,
        label=group,
        color=color,
        edgecolor='black'
    )

axes[0, 0].axvline(noise_threshold, color='black', linestyle='--', linewidth=1.5)
axes[0, 0].set_title("Compartment Magnitude")
axes[0, 0].set_xlabel(r"$||y||$")
axes[0, 0].set_ylabel("Count")
axes[0, 0].legend()

# --------------------------------------------------
# 2. Dense NNLS: number of dyes used
# --------------------------------------------------
dense_order = sorted(df_empty_compare['Unconstrained_NumDyes'].dropna().unique())

empty_dense = (
    empty_df['Unconstrained_NumDyes']
    .value_counts()
    .reindex(dense_order, fill_value=0) / max(n_empty, 1)
)

signal_dense = (
    signal_df['Unconstrained_NumDyes']
    .value_counts()
    .reindex(dense_order, fill_value=0) / max(n_signal, 1)
)

x = np.arange(len(dense_order))

axes[0, 1].bar(
    x - w/2, empty_dense.values, width=w,
    label='Empty', color='lightcoral', edgecolor='black'
)
axes[0, 1].bar(
    x + w/2, signal_dense.values, width=w,
    label='Signal', color='steelblue', edgecolor='black'
)

axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(dense_order)
axes[0, 1].set_title("Dense NNLS: Number of Dyes Used")
axes[0, 1].set_xlabel("Count of dyes with >= 5% inferred fraction")
axes[0, 1].set_ylabel("Fraction within group")
axes[0, 1].legend()

# --------------------------------------------------
# 3. BIC winner: 1-dye vs 2-dye
# --------------------------------------------------
fam_order = [1, 2]

empty_fam = (
    empty_df['BIC_BestFamily']
    .value_counts()
    .reindex(fam_order, fill_value=0) / max(n_empty, 1)
)

signal_fam = (
    signal_df['BIC_BestFamily']
    .value_counts()
    .reindex(fam_order, fill_value=0) / max(n_signal, 1)
)

x = np.arange(len(fam_order))

axes[0, 2].bar(
    x - w/2, empty_fam.values, width=w,
    label='Empty', color='lightcoral', edgecolor='black'
)
axes[0, 2].bar(
    x + w/2, signal_fam.values, width=w,
    label='Signal', color='steelblue', edgecolor='black'
)

axes[0, 2].set_xticks(x)
axes[0, 2].set_xticklabels(['1-dye', '2-dye'])
axes[0, 2].set_title("BIC Winner: 1-Dye vs 2-Dye")
axes[0, 2].set_ylabel("Fraction within group")
axes[0, 2].legend()

# --------------------------------------------------
# 4. Unconstrained NNLS R2
# --------------------------------------------------
bins_r2 = np.linspace(0, 1, 40)

axes[1, 0].hist(
    empty_df['Unconstrained_R2'].dropna(),
    bins=bins_r2, density=True,
    alpha=0.6, color='lightcoral', edgecolor='black', label='Empty'
)

axes[1, 0].hist(
    signal_df['Unconstrained_R2'].dropna(),
    bins=bins_r2, density=True,
    alpha=0.6, color='steelblue', edgecolor='black', label='Signal'
)

axes[1, 0].set_title("Unconstrained NNLS $R^2$")
axes[1, 0].set_xlabel("$R^2$")
axes[1, 0].set_ylabel("Density")
axes[1, 0].legend()

# --------------------------------------------------
# 5. BIC-selected model R2
# --------------------------------------------------
axes[1, 1].hist(
    empty_df['BIC_R2'].dropna(),
    bins=bins_r2, density=True,
    alpha=0.6, color='lightcoral', edgecolor='black', label='Empty'
)

axes[1, 1].hist(
    signal_df['BIC_R2'].dropna(),
    bins=bins_r2, density=True,
    alpha=0.6, color='steelblue', edgecolor='black', label='Signal'
)

axes[1, 1].set_title("BIC-Selected Model $R^2$")
axes[1, 1].set_xlabel("$R^2$")
axes[1, 1].set_ylabel("Density")
axes[1, 1].legend()

# --------------------------------------------------
# 6. BIC gap to runner-up
# --------------------------------------------------
gap_empty = empty_df['BIC_DeltaToSecond'].dropna()
gap_signal = signal_df['BIC_DeltaToSecond'].dropna()

gap_max = max(
    gap_empty.max() if len(gap_empty) else 0,
    gap_signal.max() if len(gap_signal) else 0
)
bins_gap = np.linspace(0, gap_max, 40) if gap_max > 0 else np.linspace(0, 1, 20)

axes[1, 2].hist(
    gap_empty,
    bins=bins_gap, density=True,
    alpha=0.6, color='lightcoral', edgecolor='black', label='Empty'
)

axes[1, 2].hist(
    gap_signal,
    bins=bins_gap, density=True,
    alpha=0.6, color='steelblue', edgecolor='black', label='Signal'
)

axes[1, 2].set_title("BIC Gap to Runner-Up")
axes[1, 2].set_xlabel(r'$\Delta$BIC = BIC$_{2nd}$ - BIC$_{best}$')
axes[1, 2].set_ylabel("Density")
axes[1, 2].legend()

plt.tight_layout()
plt.show()

## 12. Interpretation: Why “Empty” Really Is Empty

The diagnostics show that empty compartments form a distinct population:

- **Magnitude:** Empty compartments sit in a tight low-magnitude band, cleanly separated from the signal peak, so a single L2 threshold reliably gates them out.
- **Model behavior:** When forced to fit dyes, empty compartments spread tiny weights over many dyes and achieve poor R², while true signals concentrate on one (sometimes two) dyes with near-perfect fits.
- **Model selection:** BIC never finds a single, confident physical model for empties (small ΔBIC), but strongly prefers a specific one- or two-dye model for real signals.

Because empties are low-magnitude, badly explained by any dye combination, and lack a clear BIC winner, we can classify them as background noise with high confidence rather than as weak signal.

<div style="padding: 14px 16px; border-left: 8px solid #da18d3; border-radius: 8px; font-size: 1.15em; line-height: 1.4;">
    <strong style="font-size: 1.25em;">Important:</strong><br/>
    <strong>Taken together, the magnitude gate, R² fits, and BIC model selection show that truly empty compartments occupy a separate, low-intensity, poorly fitted cluster in feature space. In other words, empties behave fundamentally differently from any real dye mixture — we can know empty is empty and not signal.</strong>
</div>

## 13. Summary

### What I assumed
- Each compartment follows a linear signal model: $y = Rx + \epsilon$.
- A compartment can contain 0, 1, or 2 dyes. (this is given in the problem setup)
- Spectral overlap is expected, so we must use all channels together (not single-channel rules).


### What I decided
- First gate by signal magnitude to separate likely Empty vs likely Signal.
- Then evaluate all 1-dye and 2-dye candidates using BIC.
- Use $R^2$ as confidence checks.

### What I achieved
- Built a full end-to-end pipeline for all 10,000 compartments.
- Final assignment counts (current run/report):
  - 0-dye / Background Noise: **1,047**
  - 1-dye signal: **7,069**
  - 2-dye mixture: **1,884**
- 1-dye call breakdown (current run/report):
  - Dye 1: **913**
  - Dye 2: **930**
  - Dye 3: **815**
  - Dye 4: **583**
  - Dye 5: **980**
  - Dye 6: **988**
  - Dye 7: **929**
  - Dye 8: **931**
- Produced diagnostic plots for noise behavior, fit quality, and model confidence.

### What I observed
- Spectral overlap is present, but recoverable when using full-spectrum modeling.
- Empty compartments have distinct low-magnitude and weak-fit behavior.
- Combined gating + model fit + confidence metrics is robust and practical.


## Quick Q&A

### 1) How did I handle the spectral overlap between dyes?
- The reference spectra matrix $R$ is stably invertible, making it possible to resolve overlapping dye signals through linear algebra.

### 2) How did I separate weak signal from background noise?
- I apply a magnitude threshold first to catch likely Empty compartments.
- I then inspect fit behavior: background tends to have poorer fits and weaker model separation. This give me confidence that the empty-compartment are truly not signals.
- I use this combined evidence (magnitude + fit quality + BIC gap) to distinguish weak noise from real signal.

### 3) How did I measure confidence and flagged low-confidence calls
- **$R^2$**: how well the selected model explains the observed spectrum.